# CoolSolve — Interactive Jupyter Notebook

CoolSolve is a C++ parser, structural analyzer and equation solver for an EES-compatible language (`.eescode`), using CoolProp for thermodynamic properties.

Because the solver itself is a compiled C++ program, this notebook does not re-implement it. It **drives the real CoolSolve binary** from Python:

1. Locate / build the `coolsolve` binary
2. Run the CLI (version, parse, LaTeX output)
3. Start the built-in REST server and solve models from Python
4. Browse and solve the bundled `examples/`
5. Parametric studies with plots
6. Dynamic (`INTEGRAL`) models with trajectory plots
7. Debug mode

Run the cells top to bottom.

## 1. Setup

In [ ]:
import json, os, subprocess, sys, time, shutil, urllib.request, urllib.error, http.cookiejar
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Folder of the unzipped project (the one containing CMakeLists.txt)
PROJECT = Path("CoolSolve-main").resolve()
BUILD   = PROJECT / "build"
EXAMPLES = PROJECT / "examples"
assert (PROJECT / "CMakeLists.txt").exists(), f"Set PROJECT to the CoolSolve folder (looked in {PROJECT})"
print("Project:", PROJECT)

## 2. Build the binary

Requirements: C++17 compiler, CMake ≥ 3.14, Git, Python 3, and **Node.js 18+ with npm** (the GUI/REST server embeds the React frontend).
The first configure downloads CoolProp and other dependencies, which takes several minutes and needs internet access.

If you already have a built `coolsolve` (or the Windows installer), set `BINARY` in the next cell and skip the build.

In [ ]:
def find_binary():
    names = ["coolsolve", "coolsolve.exe"]
    for d in [BUILD, BUILD / "Release", PROJECT]:
        for n in names:
            p = d / n
            if p.exists():
                return p
    w = shutil.which("coolsolve")
    return Path(w) if w else None

BINARY = find_binary()   # or: BINARY = Path("/path/to/coolsolve")
print("Binary:", BINARY)

In [ ]:
def build(gui=True, jobs=os.cpu_count() or 2):
    """Configure and compile CoolSolve (Release). Set gui=False to skip the web server/GUI (no Node.js needed)."""
    if gui:
        # build the React frontend first so CMake can embed it
        subprocess.run(["npm", "ci"], cwd=PROJECT / "gui", check=True)
        subprocess.run(["npm", "run", "build"], cwd=PROJECT / "gui", check=True)
    BUILD.mkdir(exist_ok=True)
    subprocess.run(["cmake", "..", "-DCMAKE_BUILD_TYPE=Release",
                    f"-DCOOLSOLVE_BUILD_GUI={'ON' if gui else 'OFF'}",
                    "-DCOOLSOLVE_BUILD_TESTS=OFF"], cwd=BUILD, check=True)
    subprocess.run(["cmake", "--build", ".", "--config", "Release", "-j", str(jobs), "--target", "coolsolve"],
                   cwd=BUILD, check=True)

if BINARY is None:
    build(gui=True)
    BINARY = find_binary()
print("Using:", BINARY)

## 3. Command-line interface

In [ ]:
def cs(*args, **kw):
    """Run the coolsolve CLI and return stdout."""
    r = subprocess.run([str(BINARY), *map(str, args)], capture_output=True, text=True, **kw)
    return r.stdout + (("\n[stderr]\n" + r.stderr) if r.stderr.strip() else "")

print(cs("--version"))

In [ ]:
# Parse + structural analysis of an example, as JSON
model = EXAMPLES / "compressor_refrigeration_simple.eescode"
out = cs(model, "--no-sol")
try:
    analysis = json.loads(out)
    print("Top-level keys:", list(analysis)[:15])
except json.JSONDecodeError:
    print(out[:2000])

In [ ]:
# Equations as LaTeX
print(cs("-f", "latex", model)[:1500])

## 4. REST server from Python

`coolsolve --gui <port> --no-browser` starts the embedded web server. The class below starts it, keeps a session cookie, and wraps the endpoints documented in `docs/gui.md` (§4).

In [ ]:
class CoolSolve:
    def __init__(self, binary, port=8550):
        self.binary, self.port = binary, port
        self.base = f"http://localhost:{port}/api/v1"
        self.proc = None
        self.opener = urllib.request.build_opener(
            urllib.request.HTTPCookieProcessor(http.cookiejar.CookieJar()))

    # -- lifecycle
    def start(self, timeout=60):
        self.proc = subprocess.Popen([str(self.binary), "--gui", str(self.port), "--no-browser"],
                                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        t0 = time.time()
        while time.time() - t0 < timeout:
            try:
                h = self.get("/health")
                if h.get("status") == "ok" and h.get("coolpropReady", True):
                    return self
            except Exception:
                time.sleep(0.5)
        raise RuntimeError("CoolSolve server did not become ready (built with COOLSOLVE_BUILD_GUI=ON?)")

    def stop(self):
        if self.proc:
            self.proc.terminate()
            self.proc.wait(timeout=10)
            self.proc = None

    # -- raw HTTP
    def _req(self, method, path, body=None, timeout=300):
        data = json.dumps(body).encode() if body is not None else None
        req = urllib.request.Request(self.base + path, data=data, method=method,
                                     headers={"Content-Type": "application/json"})
        try:
            with self.opener.open(req, timeout=timeout) as r:
                txt = r.read().decode()
                return json.loads(txt) if txt else {}
        except urllib.error.HTTPError as e:
            raise RuntimeError(f"HTTP {e.code} {method} {path}: {e.read().decode()}") from e

    def get(self, p):        return self._req("GET", p)
    def post(self, p, b=None): return self._req("POST", p, b)
    def put(self, p, b):     return self._req("PUT", p, b)

    # -- model helpers
    def set_model(self, code, initials=None, conf=None):
        self.put("/files/eescode", {"content": code})
        self.put("/files/initials", {"content": initials or ""})
        if conf is not None:
            self.put("/files/conf", {"content": conf})

    def parse(self, code):
        return self.post("/parse", {"source": code})

    def _wait_stream(self, timeout=300):
        t0 = time.time()
        while time.time() - t0 < timeout:
            try:
                with self.opener.open(self.base + "/solve/stream", timeout=10) as r:
                    for raw in r:
                        line = raw.decode().strip()
                        if line.startswith("data: "):
                            try:
                                evt = json.loads(line[6:])
                            except json.JSONDecodeError:
                                continue
                            if evt.get("type") in ("done", "error"):
                                return evt
            except Exception:
                pass
            time.sleep(0.3)
        raise TimeoutError("solve did not finish")

    def solve(self, code=None, initials=None, deep_search=False, debug=False):
        """Solve and return the result dict (variables, status, blocks, timing...)."""
        body = {"deepSearch": deep_search, "debug": debug}
        if code is not None:     body["eescode"] = code
        if initials is not None: body["initials"] = initials
        self.post("/solve", body)
        self._wait_stream()
        return self.get("/solve/result")

    def variables(self, result):
        s = pd.Series(result.get("variables", {}), name="value")
        return s.sort_index().to_frame()

    def parametric(self, sweeps, code=None, timeout=600):
        """sweeps = [{"name": "T_ev", "values": [...]}, ...] (1D or 2D grid). Returns a tidy DataFrame."""
        body = {"sweepVariables": sweeps}
        if code is not None: body["eescode"] = code
        self.post("/parametric", body)
        n = 1
        for s in sweeps: n *= len(s["values"])
        t0 = time.time()
        while time.time() - t0 < timeout:
            time.sleep(0.5)
            try:
                r = self.get("/parametric/result")
            except RuntimeError:
                continue
            if r.get("totalPoints") == n and r.get("successCount", 0) + r.get("failCount", 0) == n:
                break
        else:
            raise TimeoutError("parametric study did not finish")
        rows = []
        for p in r["results"]:
            row = {**p.get("overrides", {}), "success": p["success"]}
            row.update(p.get("variables", {}))
            rows.append(row)
        return pd.DataFrame(rows)

cs_srv = CoolSolve(BINARY).start()
cs_srv.get("/health")

## 5. Solve a first model

In [ ]:
code1 = '''
"Isentropic compression of R134a"
fluid$ = 'R134a'
T_1 = 0 [C]
x_1 = 1
P_ratio = 5
h_1 = enthalpy(fluid$, T=T_1, x=x_1)
s_1 = entropy(fluid$, T=T_1, x=x_1)
P_1 = pressure(fluid$, T=T_1, x=x_1)
P_2 = P_ratio * P_1
h_2s = enthalpy(fluid$, P=P_2, s=s_1)
eta_s = 0.75
h_2 = h_1 + (h_2s - h_1) / eta_s
w = h_2 - h_1
'''
res = cs_srv.solve(code1)
print(res["status"], "| success:", res["success"], "|", res["totalTimeMs"], "ms")
cs_srv.variables(res)

### Solve a bundled example (with its `.initials` and `coolsolve.conf`)

In [ ]:
def solve_example(name, deep_search=False):
    code_ = (EXAMPLES / f"{name}.eescode").read_text(encoding="utf-8", errors="replace")
    ini_p = EXAMPLES / f"{name}.initials"
    ini   = ini_p.read_text() if ini_p.exists() else ""
    conf_p = EXAMPLES / "coolsolve.conf"
    cs_srv.set_model(code_, ini, conf_p.read_text() if conf_p.exists() else None)
    return cs_srv.solve(deep_search=deep_search)

print(sorted(p.stem for p in EXAMPLES.glob("*.eescode")))

In [ ]:
res = solve_example("compressor_refrigeration_simple")
print(res["status"], res["success"], res.get("errorMessage", ""))
df = cs_srv.variables(res)
df[df.index.str.contains("Q_dot|W_dot|COP", case=False)]

In [ ]:
# Per-block solver report
pd.DataFrame(res["blockResults"])

If a solve fails, the GUI offers **Try Harder**; the equivalent here is `deep_search=True`
(full solver pipeline, tearing and symbolic reduction forced on).

In [ ]:
# res = cs_srv.solve(deep_search=True)

## 6. Parametric study

`POST /parametric` replaces imposed assignments (`T_ev = 0`) with each sweep value and re-solves with warm starts.

In [ ]:
code_cycle = (EXAMPLES / "compressor_refrigeration_simple.eescode").read_text(encoding="utf-8", errors="replace")
cs_srv.set_model(code_cycle, "")
parse = cs_srv.parse(code_cycle)
imposed = [v["name"] for v in parse.get("variables", []) if v.get("isImposed")]
print("Imposed variables you can sweep:", imposed)

In [ ]:
# Sweep the first evaporation temperature (T_ev_1 in this example) from -20 to 10 C
sweep_var = "T_ev_1"
df_p = cs_srv.parametric([{"name": sweep_var, "values": list(range(-20, 11, 5))}])
print(df_p["success"].value_counts().to_dict())
df_p.head()

In [ ]:
ycols = [c for c in df_p.columns if c.lower().startswith(("q_dot_ev_1", "w_dot_1", "cop_1"))]
fig, ax = plt.subplots(figsize=(7, 4))
for c in ycols:
    ax.plot(df_p[sweep_var], df_p[c], marker="o", label=c)
ax.set_xlabel(sweep_var); ax.legend(); ax.grid(alpha=.3)
plt.show()

## 7. Dynamic models (`INTEGRAL`)

A first-order ODE is written in EES form `y = y0 + INTEGRAL(dydt, t, t0, tf)`. Select the integrator with the config keys `integralMethod` (`RK4`, `RK45`, `EulerExplicit`, `EulerImplicit`).

In [ ]:
code_ode = '''
"Newton cooling of a small tank"
T_amb = 20
T0 = 90
tau = 600
dTdt = -(T - T_amb) / tau
T = T0 + INTEGRAL(dTdt, t, 0, 3000)
$IntegralTable t:100, T
'''
cs_srv.set_model(code_ode, "", "integralMethod = RK45\n")
res = cs_srv.solve()
print(res["status"], res["success"], res.get("errorMessage", ""))
tab = cs_srv.get("/integral/result")
traj = pd.DataFrame(tab["data"], columns=tab["columns"]) if isinstance(tab["data"], list) and tab["data"] and isinstance(tab["data"][0], list) \
       else pd.DataFrame(tab["data"])
traj.head()

In [ ]:
xcol = tab["integrationVar"]
fig, ax = plt.subplots(figsize=(7, 4))
for c in traj.columns:
    if c != xcol:
        ax.plot(traj[xcol], traj[c], label=c)
ax.set_xlabel(xcol); ax.legend(); ax.grid(alpha=.3)
plt.show()

## 8. Debug mode

`-d` writes a folder with the variable map, equation blocks, incidence matrix, residuals, solution check, etc. (see the README for the full file list).

In [ ]:
dbg = Path("coolsolve_debug").resolve()
shutil.rmtree(dbg, ignore_errors=True)
shutil.copy(EXAMPLES / "compressor_refrigeration_simple.eescode", "model.eescode")
print(cs("-d", dbg, "model.eescode")[:500])
print(sorted(p.name for p in dbg.iterdir()))

In [ ]:
from IPython.display import Markdown
Markdown((dbg / "report.md").read_text())

# Part II — Tech-stack workflows on CoolSolve results

The sections below reuse CoolSolve output with: **CLI switches, Data Cleaning, Data Analysis, Data Visualization, Machine Learning, Deep Learning (TensorFlow), Computer Vision, Flask, HTML / CSS / Bootstrap 5 / Tailwind CSS, R, C, C++, Go and Bash**.
Every section checks that its tool is installed and skips itself otherwise. Optional installs: `pip install scikit-learn seaborn tensorflow opencv-python flask`.

## 9. Command-line switches

In [ ]:
import shlex
demo = EXAMPLES / "compressor_refrigeration_simple.eescode"
switches = {
    "--version":            ["--version"],
    "--help":               ["--help"],
    "-f latex":             ["-f", "latex", str(demo)],
    "-o FILE (json)":       ["-o", "out.json", str(demo)],
    "--no-sol":             ["--no-sol", str(demo)],
    "--no-superancillary":  ["--no-superancillary", "--no-sol", str(demo)],
    "-d DIR (debug)":       ["-d", "dbg_switch", str(demo)],
}
rows = []
for name, args in switches.items():
    t0 = time.time()
    r = subprocess.run([str(BINARY), *args], capture_output=True, text=True)
    rows.append({"switch": name, "exit_code": r.returncode, "seconds": round(time.time() - t0, 2),
                 "stdout_chars": len(r.stdout), "stderr_chars": len(r.stderr)})
pd.DataFrame(rows)

## 10. Build a dataset (2-D parametric sweep)

In [ ]:
code_cycle = (EXAMPLES / "compressor_refrigeration_simple.eescode").read_text(encoding="utf-8", errors="replace")
cs_srv.set_model(code_cycle, "")
raw = cs_srv.parametric([
    {"name": "T_ev_1", "values": [-20, -15, -10, -5, 0, 5, 10]},
    {"name": "T_cd_1", "values": [30, 35, 40, 45, 50]},
])
raw.to_csv("coolsolve_sweep_raw.csv", index=False)
print(raw.shape)
raw.head()

## 11. Data Cleaning (pandas)

In [ ]:
df = raw.copy()
df = df[df["success"] == True].drop(columns="success")          # drop failed solver points
df = df.apply(pd.to_numeric, errors="coerce")                   # force numeric
df = df.dropna(axis=1, how="all")                               # empty columns
df = df.loc[:, df.nunique() > 1].copy()                         # constant columns carry no information
df = df.drop_duplicates().reset_index(drop=True)
df.columns = [c.strip() for c in df.columns]
print("clean shape:", df.shape, "| NaNs:", int(df.isna().sum().sum()))
df.to_csv("coolsolve_sweep_clean.csv", index=False)
df.head()

## 12. Data Analysis / Data Analyst view

In [ ]:
features = [c for c in ("T_ev_1", "T_cd_1") if c in df.columns]
num = df.select_dtypes("number")
# pick target columns that look like capacity / power / COP
targets = [c for c in num.columns if c.lower().startswith(("q_dot_ev", "w_dot_", "cop")) and c not in features]
print("features:", features, "| targets:", targets[:6])
display(num[features + targets[:6]].describe().T)
corr = num[features + targets[:6]].corr()
corr

In [ ]:
# Group-by summary: effect of condensing temperature on each target
if features and targets:
    display(df.groupby(features[-1])[targets[:3]].agg(["mean", "min", "max"]).round(3))

## 13. Data Visualization (matplotlib + seaborn)

In [ ]:
tgt = targets[0] if targets else df.columns[-1]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for tcd, g in df.groupby(features[-1]):
    axes[0].plot(g[features[0]], g[tgt], marker="o", label=f"{features[-1]}={tcd:g}")
axes[0].set_xlabel(features[0]); axes[0].set_ylabel(tgt); axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)
piv = df.pivot_table(index=features[-1], columns=features[0], values=tgt)
im = axes[1].imshow(piv.values, origin="lower", aspect="auto", cmap="viridis",
                    extent=[piv.columns.min(), piv.columns.max(), piv.index.min(), piv.index.max()])
axes[1].set_xlabel(features[0]); axes[1].set_ylabel(features[-1]); fig.colorbar(im, ax=axes[1], label=tgt)
plt.tight_layout(); plt.savefig("heatmap.png", dpi=120); plt.show()
try:
    import seaborn as sns
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm"); plt.title("Correlation"); plt.show()
except ImportError:
    print("seaborn not installed (pip install seaborn)")

## 14. Machine Learning (scikit-learn surrogate model)

In [ ]:
try:
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import r2_score, mean_absolute_error
    X, y = df[features].values, df[tgt].values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
    for name, m in [("LinearRegression", LinearRegression()),
                    ("RandomForest", RandomForestRegressor(n_estimators=300, random_state=0))]:
        m.fit(Xtr, ytr); pr = m.predict(Xte)
        print(f"{name:18s} R2={r2_score(yte, pr):.4f}  MAE={mean_absolute_error(yte, pr):.4g}")
except ImportError:
    print("scikit-learn not installed (pip install scikit-learn)")

## 15. Deep Learning (TensorFlow / Keras)

In [ ]:
try:
    import numpy as np, tensorflow as tf
    X = df[features].values.astype("float32"); y = df[[tgt]].values.astype("float32")
    mu, sd = X.mean(0), X.std(0) + 1e-9; ym, ys = y.mean(), y.std() + 1e-9
    Xn, yn = (X - mu) / sd, (y - ym) / ys
    model = tf.keras.Sequential([tf.keras.layers.Input((Xn.shape[1],)),
                                 tf.keras.layers.Dense(32, activation="tanh"),
                                 tf.keras.layers.Dense(32, activation="tanh"),
                                 tf.keras.layers.Dense(1)])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-2), loss="mse")
    hist = model.fit(Xn, yn, epochs=300, batch_size=16, validation_split=0.2, verbose=0)
    plt.semilogy(hist.history["loss"], label="train"); plt.semilogy(hist.history["val_loss"], label="val")
    plt.xlabel("epoch"); plt.legend(); plt.show()
except ImportError:
    print("tensorflow not installed (pip install tensorflow)")

## 16. Computer Vision (OpenCV on the result heatmap)

In [ ]:
try:
    import cv2, numpy as np
    img = cv2.imread("heatmap.png")
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 80, 160)
    # brightest (high-value) regions of the viridis heatmap
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (20, 120, 180), (40, 255, 255))          # yellow = highest values in viridis
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = img.copy()
    for c in sorted(cnts, key=cv2.contourArea, reverse=True)[:3]:
        x, y_, w, h = cv2.boundingRect(c); cv2.rectangle(out, (x, y_), (x + w, y_ + h), (0, 0, 255), 2)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].imshow(edges, cmap="gray"); ax[0].set_title("Canny edges")
    ax[1].imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB)); ax[1].set_title("High-value regions")
    for a in ax: a.axis("off")
    plt.show()
except ImportError:
    print("opencv not installed (pip install opencv-python)")

## 17. HTML + CSS + Bootstrap 5 + Tailwind CSS report

In [ ]:
from IPython.display import HTML, display
import html as _html

def report_html(frame, title="CoolSolve sweep", framework="bootstrap"):
    table = frame.round(4).head(15).to_html(index=False, border=0, classes="table table-striped table-sm")
    if framework == "bootstrap":
        return f'''<!doctype html><html><head><meta charset="utf-8">
<link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet">
<style>.card-kpi{{border-left:6px solid #0d6efd}}</style></head><body class="p-4">
<h2>{_html.escape(title)}</h2>
<div class="row g-3 my-3">
  <div class="col-md-4"><div class="card card-kpi p-3"><div class="text-muted">Solved points</div><div class="fs-3">{len(frame)}</div></div></div>
  <div class="col-md-4"><div class="card card-kpi p-3"><div class="text-muted">Variables</div><div class="fs-3">{frame.shape[1]}</div></div></div>
</div>
<div class="table-responsive">{table}</div></body></html>'''
    return f'''<!doctype html><html><head><meta charset="utf-8">
<script src="https://cdn.tailwindcss.com/3.4.17"></script></head><body class="p-6 bg-slate-50">
<h2 class="text-2xl font-bold text-slate-800">{_html.escape(title)}</h2>
<div class="grid grid-cols-2 gap-4 my-4">
  <div class="rounded-xl bg-white shadow p-4 border-l-4 border-emerald-500"><p class="text-slate-500">Solved points</p><p class="text-3xl">{len(frame)}</p></div>
  <div class="rounded-xl bg-white shadow p-4 border-l-4 border-emerald-500"><p class="text-slate-500">Variables</p><p class="text-3xl">{frame.shape[1]}</p></div>
</div>
<div class="overflow-x-auto text-sm">{table}</div></body></html>'''

Path("report_bootstrap.html").write_text(report_html(df, framework="bootstrap"), encoding="utf-8")
Path("report_tailwind.html").write_text(report_html(df, framework="tailwind"), encoding="utf-8")
display(HTML(report_html(df, framework="bootstrap")))   # needs internet for the CDN stylesheet

## 18. Flask dashboard

In [ ]:
try:
    from flask import Flask, jsonify
    from werkzeug.serving import make_server
    import threading

    app = Flask(__name__)
    @app.get("/")
    def index():     return report_html(df, framework="bootstrap")
    @app.get("/api/data")
    def api_data():  return jsonify(df.round(6).to_dict(orient="records"))

    srv = make_server("127.0.0.1", 5055, app)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    time.sleep(1)
    with urllib.request.urlopen("http://127.0.0.1:5055/api/data") as r:
        print("Flask /api/data ->", len(json.loads(r.read())), "records  (open http://127.0.0.1:5055/)")
except ImportError:
    print("flask not installed (pip install flask)")

In [ ]:
# Stop the Flask server when finished
try: srv.shutdown()
except NameError: pass

## 19. R, C, C++, Go and Bash on the same CSV

Each program reads `coolsolve_sweep_clean.csv` and prints the mean of the first target column. Cells skip when the compiler/interpreter is missing.

In [ ]:
col_idx = list(df.columns).index(tgt) + 1       # 1-based for R / awk / cut
def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, shell=isinstance(cmd, str), **kw)
    return (r.stdout + r.stderr).strip()
def have(tool): return shutil.which(tool) is not None
print("target column:", tgt, "| index:", col_idx)

In [ ]:
# --- Bash (awk) ---
print(run(f"awk -F, 'NR>1{{s+=${col_idx};n++}}END{{print \"bash/awk mean =\", s/n}}' coolsolve_sweep_clean.csv"))

In [ ]:
# --- R ---
if have("Rscript"):
    Path("analyze.R").write_text(f'd <- read.csv("coolsolve_sweep_clean.csv")\ncat("R mean =", mean(d[[{col_idx}]]), "\\n")\nprint(summary(d[, 1:min(4, ncol(d))]))\n')
    print(run(["Rscript", "analyze.R"]))
else:
    print("Rscript not installed")

In [ ]:
# --- C ---
C_SRC = r'''
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
int main(int argc,char**argv){
  FILE*f=fopen(argv[1],"r"); int col=atoi(argv[2]); char line[1<<16]; double s=0; int n=0;
  fgets(line,sizeof line,f);
  while(fgets(line,sizeof line,f)){ char*t=strtok(line,","); for(int i=1;i<col&&t;i++) t=strtok(NULL,",");
    if(t){s+=atof(t);n++;} }
  printf("C mean = %g\n", s/n); return 0; }
'''
if have("gcc"):
    Path("mean.c").write_text(C_SRC); print(run(["gcc", "-O2", "-o", "mean_c", "mean.c"]) or run(["./mean_c", "coolsolve_sweep_clean.csv", str(col_idx)]))
else:
    print("gcc not installed")

In [ ]:
# --- C++ ---
CPP_SRC = r'''
#include <iostream>
#include <fstream>
#include <sstream>
#include <string>
int main(int argc,char**argv){
  std::ifstream f(argv[1]); int col=std::stoi(argv[2]); std::string line; std::getline(f,line);
  double s=0; int n=0;
  while(std::getline(f,line)){ std::stringstream ss(line); std::string cell;
    for(int i=1;i<=col&&std::getline(ss,cell,',');i++); s+=std::stod(cell); n++; }
  std::cout<<"C++ mean = "<<s/n<<"\n"; }
'''
if have("g++"):
    Path("mean.cpp").write_text(CPP_SRC); print(run(["g++", "-O2", "-std=c++17", "-o", "mean_cpp", "mean.cpp"]) or run(["./mean_cpp", "coolsolve_sweep_clean.csv", str(col_idx)]))
else:
    print("g++ not installed")

In [ ]:
# --- Go ---
GO_SRC = '''package main
import ("encoding/csv";"fmt";"os";"strconv")
func main(){
  f,_ := os.Open(os.Args[1]); col,_ := strconv.Atoi(os.Args[2])
  rows,_ := csv.NewReader(f).ReadAll(); s := 0.0
  for _,r := range rows[1:] { v,_ := strconv.ParseFloat(r[col-1],64); s += v }
  fmt.Printf("Go mean = %g\\n", s/float64(len(rows)-1)) }
'''
if have("go"):
    Path("mean.go").write_text(GO_SRC); print(run(["go", "run", "mean.go", "coolsolve_sweep_clean.csv", str(col_idx)]))
else:
    print("go not installed")

## 20. Cleanup

In [ ]:
cs_srv.stop()

---
**Notes**

- Full language and solver options: `docs/language_reference.md`, `docs/gui.md` (REST API), `docs/debugging_models.md`.
- Project license: MIT (see `LICENSE`).
- The helper class above targets the documented REST API of this repository version; if an endpoint's response shape differs in your build, print the raw result (`cs_srv.get("/solve/result")`) and adjust the DataFrame construction.

# Part III — 3D diagrams, charts, graphs & comparison tables

Every cell below saves a PNG into `charts/` and shows it. Data comes from the sweep (`df`, `features`, `tgt`) of sections 10–12. If the CoolSolve binary was not run, a **clearly-labelled DEMO dataset** is used so the cells still work.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from pathlib import Path
OUT = Path("charts"); OUT.mkdir(exist_ok=True)
def save(fig, name):
    fig.tight_layout(); fig.savefig(OUT / f"{name}.png", dpi=130); plt.show()

# ---- data: real sweep if available, else DEMO ----
try:
    assert len(df) > 5 and len(features) >= 2 and tgt in df.columns
    D = df.copy(); X, Y, Z = features[0], features[1], tgt; DEMO = False
except Exception:
    rng = np.random.default_rng(0)
    g = pd.MultiIndex.from_product([range(-20, 11, 5), range(30, 55, 5)], names=["T_ev_1", "T_cd_1"]).to_frame(index=False)
    Te, Tc = g.T_ev_1 + 273.15, g.T_cd_1 + 273.15
    g["COP_1"] = 0.6 * Te / (Tc - Te) + rng.normal(0, .05, len(g))
    g["W_dot_1"] = 2.0 + 0.045 * (Tc - Te) + rng.normal(0, .02, len(g))
    g["Q_dot_ev_1"] = g.COP_1 * g.W_dot_1
    D = g; X, Y, Z = "T_ev_1", "T_cd_1", "COP_1"; DEMO = True
others = [c for c in D.select_dtypes("number").columns if c not in (X, Y)][:4]
P = D.pivot_table(index=Y, columns=X, values=Z)          # grid for surfaces / bars / heatmaps
xx, yy = np.meshgrid(P.columns.values, P.index.values); zz = P.values
print("DEMO data (not solver output)" if DEMO else "CoolSolve data", "| X,Y,Z =", X, Y, Z, "| grid", zz.shape)

## A. 3D diagrams / 3D charts / 3D graphs

In [ ]:
fig = plt.figure(figsize=(8, 6)); ax = fig.add_subplot(projection="3d")
s = ax.plot_surface(xx, yy, zz, cmap="viridis", edgecolor="k", linewidth=.3)
ax.set_xlabel(X); ax.set_ylabel(Y); ax.set_zlabel(Z); ax.set_title("3D surface"); fig.colorbar(s, shrink=.6)
save(fig, "3d_surface")

In [ ]:
fig = plt.figure(figsize=(8, 6)); ax = fig.add_subplot(projection="3d")
ax.plot_wireframe(xx, yy, zz, color="tab:blue"); ax.set_xlabel(X); ax.set_ylabel(Y); ax.set_zlabel(Z); ax.set_title("3D wireframe")
save(fig, "3d_wireframe")

In [ ]:
fig = plt.figure(figsize=(8, 6)); ax = fig.add_subplot(projection="3d")
ax.contour3D(xx, yy, zz, 25, cmap="plasma"); ax.set_xlabel(X); ax.set_ylabel(Y); ax.set_zlabel(Z); ax.set_title("3D contour")
save(fig, "3d_contour")

In [ ]:
fig = plt.figure(figsize=(8, 6)); ax = fig.add_subplot(projection="3d")
ax.plot_surface(xx, yy, zz, cmap="coolwarm", alpha=.8)
ax.contourf(xx, yy, zz, zdir="z", offset=zz.min(), cmap="coolwarm")      # floor projection
ax.contourf(xx, yy, zz, zdir="x", offset=xx.min(), cmap="coolwarm", alpha=.5)
ax.contourf(xx, yy, zz, zdir="y", offset=yy.max(), cmap="coolwarm", alpha=.5)
ax.set_xlabel(X); ax.set_ylabel(Y); ax.set_zlabel(Z); ax.set_title("3D surface with projections")
save(fig, "3d_projections")

In [ ]:
fig = plt.figure(figsize=(8, 6)); ax = fig.add_subplot(projection="3d")
sc = ax.scatter(D[X], D[Y], D[Z], c=D[Z], cmap="turbo", s=40, depthshade=True)
ax.set_xlabel(X); ax.set_ylabel(Y); ax.set_zlabel(Z); ax.set_title("3D scatter"); fig.colorbar(sc, shrink=.6)
save(fig, "3d_scatter")

In [ ]:
fig = plt.figure(figsize=(8, 6)); ax = fig.add_subplot(projection="3d")
t = ax.plot_trisurf(D[X], D[Y], D[Z], cmap="magma"); ax.set_title("3D triangulated surface (scattered data)")
ax.set_xlabel(X); ax.set_ylabel(Y); ax.set_zlabel(Z); fig.colorbar(t, shrink=.6)
save(fig, "3d_trisurf")

In [ ]:
fig = plt.figure(figsize=(8, 6)); ax = fig.add_subplot(projection="3d")
for i, (yv, row) in enumerate(P.iterrows()):                           # one 3D line per Y level
    ax.plot(P.columns, [yv] * len(row), row.values, marker="o", label=f"{Y}={yv:g}")
ax.set_xlabel(X); ax.set_ylabel(Y); ax.set_zlabel(Z); ax.legend(fontsize=7); ax.set_title("3D multi-line graph")
save(fig, "3d_lines")

In [ ]:
fig = plt.figure(figsize=(8, 6)); ax = fig.add_subplot(projection="3d")
xs, ys = np.meshgrid(np.arange(zz.shape[1]), np.arange(zz.shape[0])); xs, ys = xs.ravel(), ys.ravel(); h = zz.ravel()
ax.bar3d(xs, ys, np.zeros_like(h), .7, .7, h, color=cm.viridis((h - h.min()) / (np.ptp(h) + 1e-12)), shade=True)
ax.set_xticks(np.arange(zz.shape[1]) + .35); ax.set_xticklabels([f"{v:g}" for v in P.columns], fontsize=7)
ax.set_yticks(np.arange(zz.shape[0]) + .35); ax.set_yticklabels([f"{v:g}" for v in P.index], fontsize=7)
ax.set_xlabel(X); ax.set_ylabel(Y); ax.set_zlabel(Z); ax.set_title("3D bar chart")
save(fig, "3d_bar")

In [ ]:
fig = plt.figure(figsize=(8, 6)); ax = fig.add_subplot(projection="3d")
ax.stem(D[X][:40], D[Y][:40], D[Z][:40]); ax.set_title("3D stem"); ax.set_xlabel(X); ax.set_ylabel(Y); ax.set_zlabel(Z)
save(fig, "3d_stem")

In [ ]:
fig = plt.figure(figsize=(8, 6)); ax = fig.add_subplot(projection="3d")      # 3D quiver = gradient field of Z
gy, gx = np.gradient(zz, P.index.values, P.columns.values)
ax.quiver(xx, yy, zz, gx, gy, np.zeros_like(zz), length=3, normalize=True, color="tab:red")
ax.plot_surface(xx, yy, zz, alpha=.3); ax.set_title("3D gradient arrows"); ax.set_xlabel(X); ax.set_ylabel(Y)
save(fig, "3d_quiver")

## B. Comparison tables as 3D diagrams

In [ ]:
# Pivot table (rows=Y, cols=X) drawn as a 3D bar "table"
comp = P.round(3); print(comp.to_string())
fig = plt.figure(figsize=(9, 6)); ax = fig.add_subplot(projection="3d")
for k, (yv, row) in enumerate(comp.iterrows()):
    ax.bar(np.arange(len(row)), row.values, zs=k, zdir="y", width=.6, alpha=.85, color=cm.tab10(k % 10))
ax.set_xticks(range(len(comp.columns))); ax.set_xticklabels([f"{v:g}" for v in comp.columns], fontsize=7)
ax.set_yticks(range(len(comp.index))); ax.set_yticklabels([f"{v:g}" for v in comp.index], fontsize=7)
ax.set_xlabel(X); ax.set_ylabel(Y); ax.set_zlabel(Z); ax.set_title("Comparison table as 3D grouped bars")
save(fig, "3d_comparison_bars")

In [ ]:
# Multi-metric comparison (normalised 0-1) across the first rows of the dataset, as 3D bars
M = D[[X, Y] + others[:3]].head(6).reset_index(drop=True)
vals = M[others[:3]]; N = (vals - vals.min()) / (vals.max() - vals.min() + 1e-12)
fig = plt.figure(figsize=(9, 6)); ax = fig.add_subplot(projection="3d")
for j, c in enumerate(N.columns):
    ax.bar(np.arange(len(N)), N[c].values, zs=j, zdir="y", width=.6, alpha=.85, color=cm.Set2(j))
ax.set_yticks(range(len(N.columns))); ax.set_yticklabels(N.columns, fontsize=7)
ax.set_xticks(range(len(N))); ax.set_xticklabels([f"{a:g}/{b:g}" for a, b in zip(M[X], M[Y])], fontsize=7, rotation=20)
ax.set_zlabel("normalised"); ax.set_title("Multi-metric comparison (3D)")
save(fig, "3d_comparison_metrics")

In [ ]:
# Flat table image + styled HTML table
fig, ax = plt.subplots(figsize=(9, 2 + .35 * len(comp))); ax.axis("off")
t = ax.table(cellText=comp.values, rowLabels=[f"{v:g}" for v in comp.index], colLabels=[f"{v:g}" for v in comp.columns],
             cellColours=cm.YlGnBu((comp.values - np.nanmin(comp.values)) / (np.ptp(comp.values) + 1e-12)), loc="center")
t.auto_set_font_size(False); t.set_fontsize(8); ax.set_title(f"{Z}: rows={Y}, cols={X}")
save(fig, "table_image")
comp.style.background_gradient(cmap="viridis").format("{:.3f}")

## C. All the standard charts & graphs (2D)

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(15, 12)); a = axs.ravel()
for yv, row in P.iterrows(): a[0].plot(P.columns, row.values, marker="o", label=f"{yv:g}")
a[0].set_title("Line"); a[0].legend(fontsize=6, title=Y); a[0].set_xlabel(X); a[0].set_ylabel(Z)
mid = P.iloc[len(P)//2]; a[1].bar([f"{v:g}" for v in P.columns], mid.values, color="tab:orange"); a[1].set_title(f"Bar ({Y}={P.index[len(P)//2]:g})")
w = .8 / len(P); 
for i, (yv, row) in enumerate(P.iterrows()): a[2].bar(np.arange(len(row)) + i * w, row.values, w, label=f"{yv:g}")
a[2].set_title("Grouped bar")
bot = np.zeros(len(P.columns))
for yv, row in P.iterrows(): a[3].bar([f"{v:g}" for v in P.columns], row.values, bottom=bot); bot += row.values
a[3].set_title("Stacked bar")
a[4].stackplot(P.columns, P.values, labels=[f"{v:g}" for v in P.index]); a[4].set_title("Stacked area")
a[5].scatter(D[X], D[Z], c=D[Y], cmap="viridis"); a[5].set_title("Scatter (colour = " + Y + ")")
a[6].hist(D[Z], bins=12, color="tab:green", edgecolor="k"); a[6].set_title("Histogram")
a[7].boxplot([r.values for _, r in P.iterrows()]); a[7].set_xticklabels([f"{v:g}" for v in P.index]); a[7].set_title("Box plot")
a[8].violinplot([r.values for _, r in P.iterrows()]); a[8].set_title("Violin plot")
save(fig, "2d_gallery_1")

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(15, 12)); a = axs.ravel()
a[0].pie(mid.values, labels=[f"{v:g}" for v in P.columns], autopct="%1.0f%%"); a[0].set_title("Pie")
a[1].pie(mid.values, labels=[f"{v:g}" for v in P.columns], wedgeprops=dict(width=.4)); a[1].set_title("Donut")
im = a[2].imshow(zz, origin="lower", aspect="auto", cmap="inferno", extent=[xx.min(), xx.max(), yy.min(), yy.max()]); fig.colorbar(im, ax=a[2]); a[2].set_title("Heatmap")
c = a[3].contourf(xx, yy, zz, 15, cmap="viridis"); fig.colorbar(c, ax=a[3]); a[3].set_title("Filled contour")
a[4].hexbin(D[X], D[Y], C=D[Z], gridsize=8, cmap="plasma"); a[4].set_title("Hexbin")
a[5].errorbar(P.columns, P.mean(0), yerr=P.std(0), fmt="o-", capsize=4); a[5].set_title("Error bars (mean ± std over " + Y + ")")
a[6].step(P.columns, mid.values, where="mid"); a[6].set_title("Step")
a[7].stem(P.columns, mid.values); a[7].set_title("Stem")
a[8].fill_between(P.columns, P.min(0), P.max(0), alpha=.3); a[8].plot(P.columns, P.mean(0)); a[8].set_title("Range band")
save(fig, "2d_gallery_2")

In [ ]:
fig = plt.figure(figsize=(15, 5))
ax = fig.add_subplot(131, projection="polar")                           # polar
th = np.linspace(0, 2 * np.pi, len(mid), endpoint=False); ax.plot(np.r_[th, th[0]], np.r_[mid.values, mid.values[0]]); ax.fill(th, mid.values, alpha=.3); ax.set_title("Polar")
ax = fig.add_subplot(132, projection="polar")                           # radar of normalised metrics
cols = others[:4] or [Z]; r = D[cols].mean(); r = (r - D[cols].min()) / (D[cols].max() - D[cols].min() + 1e-12)
th = np.linspace(0, 2 * np.pi, len(cols), endpoint=False); ax.plot(np.r_[th, th[0]], np.r_[r, r.iloc[0]]); ax.fill(th, r, alpha=.3)
ax.set_xticks(th); ax.set_xticklabels(cols, fontsize=7); ax.set_title("Radar")
ax = fig.add_subplot(133); cm_ = D.select_dtypes("number").corr().iloc[:8, :8]
im = ax.imshow(cm_, cmap="coolwarm", vmin=-1, vmax=1); ax.set_xticks(range(len(cm_))); ax.set_xticklabels(cm_.columns, rotation=60, fontsize=6)
ax.set_yticks(range(len(cm_))); ax.set_yticklabels(cm_.index, fontsize=6); fig.colorbar(im, ax=ax); ax.set_title("Correlation matrix")
save(fig, "2d_gallery_3")

In [ ]:
try:
    import seaborn as sns
    sns.pairplot(D[[X, Y] + others[:2]]); plt.savefig(OUT / "pairplot.png", dpi=110); plt.show()
except ImportError:
    print("pip install seaborn for the pair plot")

In [ ]:
# Summary tables
display(D.describe().T.round(3))
display(D.groupby(Y)[[Z] + others[:2]].agg(["mean", "min", "max"]).round(3))
print("PNG files:", sorted(p.name for p in OUT.glob("*.png")))